<div style="background-color: lightblue; padding: 10px; border-radius: 5px;">

## From Molecules to Graph Neural Networks (GNN) - No Libraries

This notebook demonstrates how to build a simple (GNN) from scratch using `PyTorch`, without relying on libraries such as `PyG`.

We begin by representing small molecules as graphs (using either hard coded arrays or `RDKit`): atoms become nodes, and bonds become edges. We then construct the node features and adjacency structure manually, and use basic tensor operations to implement message passing and pooling.

The goal is to learn how GNNs operate at the tensor level, before transitioning to library-based implementations.


<div style="background-color: lightblue; padding: 10px; border-radius: 5px;">
    
**Training datasets** : 

* a few molecules already featurized ad hoc into their graph representation, to be used a train set and test set: hard-coded
* a few molecules represented by SMILES strings; RDKit is used to featurize them

Here, we want to train a GNN classifier that can tell whether a molecule contains Oxygen (label=1) or not (label=0)
</div>

<div style="background-color: lightblue; padding: 10px; border-radius: 5px;">
    
**Tools**
* `scikit-learn`, `torch`, `networkx`
* **Core cheminfo**: `RDKit` **smiles, descriptors** to automate molecular representation and generate the input to the classifier
* `torch` to define the GNN layers [message passing + aggregation, connectivity is never changed], the head of network is a classifier; train and test
  - I wrote a `FlexibleGNNLayer` layer that uses either node-only or node+edge message passing, depending on the inputs. As of now, messages are pooled by summing the up together, 101 pooling choice
</div>

In [ ]:
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F   # these are the activation functions
from torchinfo import summary

# Set random seed for reproducibility
torch.manual_seed(42)

from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem import rdmolops
import networkx as nx
from IPython.display import display

import matplotlib
import matplotlib.pyplot as plt

try:
    from src.utils import mol_to_nx, visualize_molecular_graph, atom_features, bond_features
    from src.utils import GraphClassifier
except ImportError:
    print("Missing src/utils module. Make sure it's in the correct path.")

<div style="background-color: lightgreen; padding: 10px; border-radius: 5px;">
    
### First part: use hard-coded molecular featurizations
</div>

<div style="background-color: lightgreen; padding: 10px; border-radius: 5px;">
    
#### Trainset: 3 molecules (C-C-O, C-C-H, H-O), whose graph representations are hard coded. The task is to classify whether each molecule (graph-based task) contains any oxigen atoms
</div>

In [ ]:
# Hard-coded toy molecules: GRAPH FEATURIZATION

# === FEATURES, aka "X" ========#
# Atom type to one-hot vector: C, O, H ... dictionary: the size of those vectors is a parameter to the model
atom_type_to_vec = { 'C': [1, 0, 0, 0], 'O': [0, 1, 0, 0], 'H': [0, 0, 1, 0], 'S': [0, 0, 0, 1], 'K': [2, 0, 0, 0]}

node_dim_feat = 4 # size of the node feature vector

# === Molecule 1: C—C—O (label 1: contains oxygen):
# node embeddings
mol1_nodes = torch.tensor([ atom_type_to_vec['C'], atom_type_to_vec['C'], atom_type_to_vec['O']], dtype=torch.float)
# edge embeddings (please note the symmetry)
mol1_edges = torch.tensor([[0, 1],  [1, 0], [1, 2],  [2, 1]])

# === Molecule 2: C—C—H (label 0: no oxygen)
mol2_nodes = torch.tensor([ atom_type_to_vec['C'],  atom_type_to_vec['C'],   atom_type_to_vec['H'] ], dtype=torch.float)
mol2_edges = torch.tensor([  [0, 1],    [1, 0], [1, 2],    [2, 1]])

# === Molecule 3: H—O (label 1: yes oxygen)
mol3_nodes = torch.tensor([atom_type_to_vec['H'], atom_type_to_vec['O']], dtype=torch.float)
mol3_edges = torch.tensor([   [0, 1],    [1, 0]])

assert mol1_nodes.shape[1] == mol2_nodes.shape[1] == mol3_nodes.shape[1] ==node_dim_feat     # are all nodes correctly embedded?

# this example only has nodes, node attributes and edges, no edge attribute. Here, edge attributes are mute variables, are not propagated.
# EXTENSION IDEA: add edge attr to see if classification improves

# ==== Labels, aka 'y' =====
# Labels: 1 if contains oxygen, else 0, this is the trainign data
labels = torch.tensor([1, 0, 1], dtype=torch.float)

<div style="background-color: lightgreen; padding: 10px; border-radius: 5px;">
    
#### Instantiate and train GNN classifier that is build from scratch and imported from `utils.py`

#### We define a simple 2-layer GNN model using PyTorch. This model operates over node features and adjacency information directly, mimicking message passing by matrix multiplication followed by non-linearity and pooling (node message passing only)


</div>

In [ ]:
# the neural network has been imported from the utils file
model = GraphClassifier(node_in_dim=node_dim_feat, edge_in_dim = None, hidden_dim=10)    # each node comes with a [x,y,z, t] 3D embedding
print(model)

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)   # optimizer
loss_fn = nn.BCEWithLogitsLoss()      # binary cross entropy with logit: need to use sigmoid on the logits at the end to get predictions

print('===== model architecture ===')
print(model)
print('\n')
print('start trainng...')

for epoch in range(100):
    model.train()
    optimizer.zero_grad()    ### initialize the gradients for backpropagation

    pred1 = model(mol1_nodes, mol1_edges)    # run this on molecule 1
    pred2 = model(mol2_nodes, mol2_edges)    # tun this on molecule 2
    pred3 = model(mol3_nodes, mol3_edges)    # tun this on molecule 2
    
    logits = torch.cat([pred1, pred2, pred3], dim=0)   # concatenate (torch tensor)
    loss = loss_fn(logits, labels)          # preds are from the input, labels are the known labels (torch tensor)

    loss.backward()
    optimizer.step()

    if epoch % 10 == 0 or epoch == 99:
        with torch.no_grad():
            pred_class = (torch.sigmoid(logits) > 0.5).int()
            acc = (pred_class == labels.int()).float().mean()
            print(f"Epoch {epoch:3d} | Loss: {loss.item():.4f} | Accuracy: {acc:.2f}")

print('GNN classifier model is trained!')
summary(model, input_data=(mol1_nodes, mol1_edges))

<div style="background-color: lightgreen; padding: 10px; border-radius: 5px;">
    
#### check predictions on the input, just for benchmarking
</div>

In [ ]:
# check if performance on training set makes sense

with torch.no_grad():     # Useful during inference, evaluation, or any time you don’t need to update model parameters.
                          # predictions and assessments do not require storing gradient information
    print("\nFinal predictions:")
    print(f"Molecule 1 (C-C-O): {torch.sigmoid(model(mol1_nodes, mol1_edges)).item():.3f}")
    print(f"Molecule 2 (C-C-H): {torch.sigmoid(model(mol2_nodes, mol2_edges)).item():.3f}")
    print(f"Molecule 3 (H-0): {torch.sigmoid(model(mol3_nodes, mol3_edges)).item():.3f}")
    
# idea: use a dictionary to loop over the different input moleuclar graphs to be processed

<div style="background-color: lightgreen; padding: 10px; border-radius: 5px;">
    
#### Test trained GNN optimizer on a novel molecule the user can hard-code here by giving its graph representation
</div>

In [ ]:
# define a new test molecule 
test_mol_nodes = torch.tensor([
    atom_type_to_vec['H'],     # we are featurizing the atom label, so it is agnostic. Otherwise, one would just look at the smile rep.
    atom_type_to_vec['H'],
    atom_type_to_vec['H'],
    atom_type_to_vec['H'],
    atom_type_to_vec['H']], dtype=torch.float)

test_mol_edges = torch.tensor([  [0, 1], [1, 0], [1, 2],  [2, 1],  [0, 2],  [2, 0]])

#and run classifier on it
etichetta = torch.sigmoid(model(test_mol_nodes, test_mol_edges)).item()

if etichetta == 1:
    print('Molecule does contain Oxygen')
else:
    print('Molecule does not contain Oxygen')


<div style="background-color: lightgreen; padding: 10px; border-radius: 5px;">
    
### Second part: use RDKit for graph featurization
</div>

In [ ]:
def graph_featurizer(mol):

    """A molecule is translated into a featurized graphs, with nodes and node labels (5-dim array) + edges and one-hotedge 
    labels (4-dim array)

    INPUT: * mol, RDKit object
    OUTPUT: * x: torch tensor, (n_samples, 5)   feaures of all the nodes
            * edge_index: torch tensor, (n_bonds * 2, 2)  # adjacency list of the edges
            * edge_attr: torch tensor, (n_bonds * 2, 4)   features of all the edges
    """
    
    atom_feats = []
    edge_index = []
    edge_attr = []
    
    for atom in mol.GetAtoms():
        atom_feats.append(atom_features(atom))     # list of torch tensors
    
    for bond in mol.GetBonds():
            i = bond.GetBeginAtomIdx()
            j = bond.GetEndAtomIdx()
            edge_index.append([i, j])
            edge_index.append([j, i])  # graph is undirected: this ensure symmetric message passing
            edge_attr.append(bond_features(bond))   
            edge_attr.append(bond_features(bond)) # if bidirectional, we miss half the bonds if we don't include the flipped one
                                                  # clearly here [i,j] and [j,i] share the same feature; 
    # from list of torch tensors to one torch tensor
    x = torch.stack(atom_feats)            
    edge_attr = torch.stack(edge_attr)
    
    # from a list of numpy arrays to a torch tensor
    edge_index = torch.tensor(edge_index, dtype=torch.long)   # forget about transposing this, as we are testing
    
    #print(x.shape)    # (number of atoms, number of features)
    #print(edge_index.shape)    # (2, number of edges * 2), each edge is listed twice (i,j and j,i)
    #print(edge_attr.shape)    # (number of edges * 2, size of one-hot encoding
    
    return x, edge_index, edge_attr
    
# NB these features I would stack in a pytorch tensors and those would be good to be compatible with PyTorch Geometric

In [ ]:
# === Step 1: SMILES → Molecola RDKit ===
smiles_list = [
    "CCO",         # Etanolo
    "CC(=O)O",     # Acido acetico
    "c1ccccc1",    # Benzene
    "CCN(CC)CC",    # Trietilammina
    "CC(O)C" , 
    'CC(CCC(=O)N)CN'
]

# extract mol RDKit object from smiles stsring
mol_list = [Chem.MolFromSmiles(smi) for smi in smiles_list] 

In [ ]:
mol_idx = 3
print(smiles_list[mol_idx])
visualize_molecular_graph(mol_to_nx(mol_list[mol_idx]), node_size = 860, node_color = 'lightblue', smiles = smiles_list[mol_idx])

In [ ]:
# === Step 2: RDKit allows to visualize the diagram/representation of the molecule ===
img = Draw.MolsToGridImage(mol_list, molsPerRow=6, subImgSize=(200, 200))
display(img)

# compute adjacency matrix for the molecules
#for k, mol in enumerate(mol_list):
#    print(smiles_list[k], rdmolops.GetAdjacencyMatrix(mol))

In [ ]:
# need to hard code the ouput (one could also extract it from lookin at the smiles strings"
labels = torch.tensor([1,1,0,0,1, 1], dtype=torch.float)

# featurize all molecules using nodes and edges and edge features
dataset = [graph_featurizer(mol) for mol in mol_list]
print(dataset[1])

node_dim_feat = dataset[0][0].shape[1]
edge_dim_feat = dataset[0][2].shape[1]
n_samples = len(dataset)

print(f'Number of samples to be used in training = {n_samples}')
print(f'Size of node feature embeddings = {node_dim_feat}')
print(f'Size of edge feature embeddings = {edge_dim_feat}')

In [ ]:
# define model and optimizer
model = GraphClassifier(node_in_dim=node_dim_feat, edge_in_dim = edge_dim_feat,hidden_dim=10)    # each node comes with a [x,y,z, t] 3D embedding
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)   # optimizer
loss_fn = nn.BCEWithLogitsLoss()      # binary cross entropy with logit: need to use sigmoid on the logits at the end to get predictions
## nn.MSELoss(), self.regressor(nn.Linear(hidden_dim, 1))

print('===== model architecture ===')
print(model)
print('\n')

n_epochs = 100
      
# === STEP 5: Training loop ===
for epoch in range(n_epochs):
    model.train()
    optimizer.zero_grad()    ### initialize the gradients for backpropagation

    predictions = []
    for n in range(n_samples):
        predictions.append(model(node_feats = dataset[n][0], edge_index = dataset[n][1], edge_attr = dataset[n][2]))
    logits = torch.cat(predictions, dim = 0)
    
    loss = loss_fn(logits, labels)          # preds are from the input, labels are the known labels (torch tensor)
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0 or epoch == 99:
        with torch.no_grad():
            pred_class = (torch.sigmoid(logits) > 0.5).int()
            acc = (pred_class == labels.int()).float().mean()
            print(f"Epoch {epoch:3d} | Loss: {loss.item():.4f} | Accuracy: {acc:.2f}")

In [ ]:
# check how trained model behaves on the training data (sanity check)

with torch.no_grad():     # Useful during inference, evaluation, or any time you don’t need to update model parameters.
                          # predictions and assessments do not require storing gradient information
    print("\nFinal predictions:")

    for n in range(n_samples):
        x, edges, attrib = graph_featurizer(mol_list[n])
        print(f"Molecule {n} {smiles_list[n]}: {torch.sigmoid(model(x, edges, attrib)).item():.3f}")

In [ ]:
# check how the model behaves on a test molecule (any SMILES string)

test_smile = 'CC(C)CO'
test_mol = Chem.MolFromSmiles(test_smile)

x, edges, attrib = graph_featurizer(test_mol)
print(x)

print(f"Molecule {test_smile}: {torch.sigmoid(model(x, edges, attrib)).item():.3f}")

<div style="background-color: lightyellow; padding: 10px; border-radius: 5px;">
    
## Summary

In this notebook, we represented molecular structures as graphs and implemented a simple GNN from scratch using only `PyTorch`.

This helped demonstrate:
- How molecules can be encoded as node + edge structures
- How message passing works and can be simulated 
- How to make predictions over graph-level labels

In future notebooks, we will build on this foundation using `PyG` for more complex architectures and scalable batching across molecules.

</div>